# Startup Model Workbench

This notebook trains and compares three models on `data/startup_funding_and_outcome.csv`, plots ROC/AUC curves, reports the core metrics, picks the best model, and saves the final pipeline as a `.pkl` file for the app to load.

The runtime app uses the saved artifact through `src/ml/predictor.py`; this notebook is the training and evaluation workflow.

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current.parent, current.parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "startup_funding_and_outcome.csv").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "startup_funding_and_outcome.csv"
MODEL_PATH = PROJECT_ROOT / "src" / "ml" / "models" / "startup_best_model.pkl"

FEATURE_COLUMNS = [
    "funding_rounds",
    "founder_experience_years",
    "team_size",
    "market_size_billion",
    "product_traction_users",
    "burn_rate_million",
    "revenue_million",
    "investor_type",
    "sector",
    "founder_background",
]
NUMERIC_FEATURES = [
    "funding_rounds",
    "founder_experience_years",
    "team_size",
    "market_size_billion",
    "product_traction_users",
    "burn_rate_million",
    "revenue_million",
]
CATEGORICAL_FEATURES = ["investor_type", "sector", "founder_background"]
TARGET_COLUMN = "outcome"

def map_outcome(value):
    text = str(value).strip().lower()
    return 1 if text in {"ipo", "acquisition", "success", "1", "operating"} else 0

def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = set(FEATURE_COLUMNS + [TARGET_COLUMN])
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    return df

def build_preprocessor() -> ColumnTransformer:
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    numeric_pipe = Pipeline([
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("encoder", encoder),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipe, NUMERIC_FEATURES),
        ("categorical", categorical_pipe, CATEGORICAL_FEATURES),
    ])

def make_pipeline(model) -> Pipeline:
    return Pipeline([
        ("preprocess", build_preprocessor()),
        ("model", model),
    ])

In [ ]:
df = load_data(DATA_PATH)
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].apply(map_outcome)

print(f"Loaded {len(df):,} rows from {DATA_PATH.name}")
print("Target balance:")
display(y.value_counts().rename({0: "Failure", 1: "Success"}).to_frame("count"))
display(df.head())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train set: {X_train.shape[0]:,} rows | Test set: {X_test.shape[0]:,} rows")

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced_subsample"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
trained_models = {}
roc_curves = {}
reports = {}

for model_name, estimator in models.items():
    pipeline = make_pipeline(estimator)
    cv_auc = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)

    metrics = {
        "model": model_name,
        "cv_auc_mean": cv_auc.mean(),
        "cv_auc_std": cv_auc.std(),
        "test_auc": roc_auc_score(y_test, y_prob),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    }

    results.append(metrics)
    trained_models[model_name] = pipeline
    roc_curves[model_name] = {
        "fpr": fpr,
        "tpr": tpr,
        "auc": metrics["test_auc"],
    }
    reports[model_name] = classification_report(y_test, y_pred, output_dict=True)

results_df = pd.DataFrame(results).sort_values("test_auc", ascending=False).reset_index(drop=True)
display(results_df)

best_model_name = results_df.loc[0, "model"]
best_model = trained_models[best_model_name]
print(f"Best model by test ROC AUC: {best_model_name}")
display(pd.DataFrame(reports[best_model_name]).T)

In [ ]:
plt.figure(figsize=(9, 7))
for model_name, curve in roc_curves.items():
    plt.plot(curve["fpr"], curve["tpr"], linewidth=2, label=f"{model_name} (AUC = {curve['auc']:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
plt.title("ROC Curves Across Candidate Models")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.25)
plt.show()

best_curve = roc_curves[best_model_name]
plt.figure(figsize=(7, 5))
plt.plot(best_curve["fpr"], best_curve["tpr"], linewidth=2.5, label=f"{best_model_name} (AUC = {best_curve['auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
plt.title(f"Best Model ROC Curve: {best_model_name}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.25)
plt.show()

In [ ]:
final_pipeline = make_pipeline(models[best_model_name])
final_pipeline.fit(X, y)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, MODEL_PATH)

print(f"Saved best model pipeline to: {MODEL_PATH}")
print("Notebook run complete. The app can now load the saved pkl through src/ml/predictor.py.")

## What to do next

- Re-run this notebook whenever you want to retrain the model.
- The saved artifact is `src/ml/models/startup_best_model.pkl`.
- The FastAPI app uses `src/ml/predictor.py` to load the pkl and score a startup from the UI inputs.